# 32_gold_ml — Gold ML Feature Tables

Builds three ML feature tables from `silver.events` + `silver.tracks` (using business keys, not SKs).
Full recompute with `overwrite` on each run (idempotent by construction).

**Tables:**
- `gold.ml_user_track_interactions` — user × track interaction matrix
- `gold.ml_track_features` — track audio features + cross-user engagement
- `gold.ml_user_sessions` — session-level feature table

> `real_api` rows have NULL `ms_played`/`completion_rate` — excluded from completion-based features.

In [ ]:
%run ./00_config


## ML-1: User × Track Interaction Matrix

In [ ]:
from datetime import datetime, timezone

_t0 = datetime.now(timezone.utc)
try:
    df_int = spark.sql(f"""
        SELECT
            user_id,
            track_id,
            COUNT(*)                                                   AS play_count,
            SUM(ms_played)                                             AS total_ms_played,
            AVG(CASE WHEN data_source <> 'real_api' THEN completion_rate END) AS avg_completion,
            MIN(played_at)                                             AS first_played,
            MAX(played_at)                                             AS last_played,
            CURRENT_TIMESTAMP()                                        AS load_timestamp
        FROM {table_ref(SILVER_EVENTS)}
        GROUP BY user_id, track_id
    """)
    n = df_int.count()
    df_int.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['ml_user_track_interactions'])
    log_run('SILVER_TO_GOLD','gold.ml_user_track_interactions','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'ml_user_track_interactions: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.ml_user_track_interactions','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## ML-2: Track Feature Table

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_tf = spark.sql(f"""
        SELECT
            e.track_id,
            COUNT(*)                                                      AS total_plays,
            COUNT(DISTINCT e.user_id)                                     AS unique_listeners,
            AVG(CASE WHEN e.data_source <> 'real_api' THEN e.completion_rate END)
                                                                          AS avg_completion,
            AVG(CAST(e.skipped AS INT))                                   AS skip_rate,
            t.danceability,
            t.energy,
            t.valence,
            t.tempo_bpm,
            t.popularity,
            t.genre,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM {table_ref(SILVER_EVENTS)} e
        LEFT JOIN {table_ref(SILVER_TRACKS)} t ON e.track_id = t.track_id
        GROUP BY e.track_id, t.danceability, t.energy, t.valence,
                 t.tempo_bpm, t.popularity, t.genre
    """)
    n = df_tf.count()
    df_tf.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['ml_track_features'])
    log_run('SILVER_TO_GOLD','gold.ml_track_features','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'ml_track_features: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.ml_track_features','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## ML-3: User Session Feature Table

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_sess = spark.sql(f"""
        SELECT
            session_id,
            user_id,
            MIN(played_at)               AS session_start,
            MAX(played_at)               AS session_end,
            COUNT(*)                     AS track_count,
            COUNT(DISTINCT t.genre)      AS genre_diversity,
            MODE(e.device_type)          AS dominant_device,
            CURRENT_TIMESTAMP()          AS load_timestamp
        FROM {table_ref(SILVER_EVENTS)} e
        LEFT JOIN {table_ref(SILVER_TRACKS)} t ON e.track_id = t.track_id
        GROUP BY session_id, user_id
    """)
    n = df_sess.count()
    df_sess.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['ml_user_sessions'])
    log_run('SILVER_TO_GOLD','gold.ml_user_sessions','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'ml_user_sessions: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.ml_user_sessions','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Proof: ML Table Row Counts

In [ ]:
for t in ['ml_user_track_interactions','ml_track_features','ml_user_sessions']:
    print(f'{GOLD[t]}: {spark.table(GOLD[t]).count():,} rows')
